<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/SimuladorPlan_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# MÓDULO INDEPENDIENTE: SIMULADOR DE ESCENARIOS
# ==============================================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display

print("Iniciando Módulo de Simulación de Escenarios Multi-Horizonte...")

def preparar_datos_ventas(archivo_ventas, dias_visibilidad=7):
    """
    Función auxiliar para cargar y preparar la demanda comercial.
    """
    try:
        df_ventas = pd.read_excel(archivo_ventas)
        df_ventas = df_ventas.drop(0)
        df_ventas.columns = ["OrdenEPS","Año","Sem","Bloque","Critico","Cliente","Ciudad","Producto Terminado",
                     "Fecha","Real","Componente","Ing","Tipo","Transf","Proceso","Pzas","Hrs",
                     "Nesteada","Cls","Placa","Esp","CodigoRadan","Color pintura","Peso","Familia",
                     "Color","Comentario","Nesteos","idNesteos","TiempoCorteHrs","CorteLaser","Plasma","Nivelado","Pulidor","Limpieza","quintar filos",
                     "Inspeccion","Etiqueta especial"]

        df_ventas['Real'] = pd.to_datetime(df_ventas['Real'])

        # Calcular horizonte de urgencia
        hoy = datetime.now()
        fecha_limite = hoy + timedelta(days=dias_visibilidad)

        df_ventas_urgentes = df_ventas[df_ventas['Real'] <= fecha_limite].copy()
        print(f"✅ Ventas cargadas: {len(df_ventas_urgentes)} líneas críticas detectadas.")
        return df_ventas, df_ventas_urgentes

    except FileNotFoundError:
        print(f"⚠️ Error: No se encontró el archivo de ventas '{archivo_ventas}'.")
        return None, None

def simular_cobertura_escenarios(archivo_plan_actual, df_ventas, df_ventas_urgentes, horizontes=[10, 24, 48, 72, 96, 120]):
    """
    Simula el plan del ERP de una sola vez y extrae la proyección para múltiples escenarios de horas.
    """
    if df_ventas is None or df_ventas_urgentes is None:
        return None

    print(f"\nArrancando simulación de reloj maestro para escenarios de {horizontes} horas...")

    try:
        # =========================================================
        # 1. Cargar el plan, eliminar encabezado basura y estandarizar
        # =========================================================
        df_plan = pd.read_excel(archivo_plan_actual)
        df_plan = df_plan.drop(0)

        df_plan.columns = [
            "No", "Maquina", "Folio", "Programa", "Material", "CantPlacas", "Surtido",
            "en proceso", "bloque", "Numero de parte", "Cant_piezas", "ruta de proceso",
            "Corte", "Setup", "iniciocorte", "fincorte", "acum", "nest", "Fecha",
            "disponible", "contenedor", "terminado"
        ]

        df_plan['No'] = pd.to_numeric(df_plan['No'], errors='coerce')
        df_plan['Corte'] = pd.to_numeric(df_plan['Corte'], errors='coerce') * 60.0
        df_plan['Cant_piezas'] = pd.to_numeric(df_plan['Cant_piezas'], errors='coerce').fillna(0)

        df_plan['Folio'] = df_plan['Folio'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        df_plan['Maquina'] = df_plan['Maquina'].astype(str).str.strip()
        df_plan['Material'] = df_plan['Material'].astype(str).str.strip()
        df_plan['Numero de parte'] = df_plan['Numero de parte'].astype(str).str.strip()

        df_plan = df_plan.dropna(subset=['No', 'Folio'])
        df_plan = df_plan.sort_values(by=['Maquina', 'No'])

        # =========================================================
        # 2. Simular el "Reloj" de la Planta UNA SOLA VEZ
        # =========================================================
        resultados_sim = []
        for maquina, grupo_maquina in df_plan.groupby('Maquina', sort=False):
            reloj = 0.0
            material_previo = None

            for folio, grupo_folio in grupo_maquina.groupby('Folio', sort=False):
                row_folio = grupo_folio.iloc[0]
                material_actual = row_folio['Material']
                tiempo_corte = row_folio['Corte']

                setup = 15.0 if material_previo is not None and material_actual != material_previo else 0.0

                inicio = reloj + setup
                fin = inicio + tiempo_corte

                for idx, row_pieza in grupo_folio.iterrows():
                    resultados_sim.append({
                        'Folio': folio,
                        'Maquina': maquina,
                        'Inicio (min)': inicio,
                        'Fin (min)': fin,
                        'Numero de parte': row_pieza['Numero de parte'],
                        'Piezas_Simuladas': row_pieza['Cant_piezas']
                    })
                reloj = fin
                material_previo = material_actual

        df_reloj_simulado = pd.DataFrame(resultados_sim)

        # =========================================================
        # 3. Preparar Base para Gráfica y Analizar Escenarios
        # =========================================================
        demanda_cliente = df_ventas_urgentes.groupby('Cliente')['Pzas'].sum().reset_index()
        demanda_cliente.rename(columns={'Pzas': 'Demanda_Critica'}, inplace=True)
        df_plot_all = demanda_cliente.copy()

        colores_escenarios = {8: '#e67e22', 16: '#f1c40f', 24: '#3498db', 48: '#2ecc71', 72: '#9b59b6', 96: '#e74c3c', 120: '#34495e'}

        for horas in horizontes:
            minutos_limite = horas * 60.0
            df_dentro_turno = df_reloj_simulado[df_reloj_simulado['Fin (min)'] <= minutos_limite]
            produccion_simulada = df_dentro_turno.groupby('Numero de parte')['Piezas_Simuladas'].sum().to_dict()

            df_ventas_detalle = df_ventas.sort_values(by=['Componente', 'Real']).copy()
            lineas_simuladas = []

            for idx, row in df_ventas_detalle.iterrows():
                comp = row['Componente']
                pedidas = row['Pzas']

                if comp in produccion_simulada and produccion_simulada[comp] > 0:
                    surtido = min(pedidas, produccion_simulada[comp])
                    produccion_simulada[comp] -= surtido

                    lineas_simuladas.append({
                        'Cliente': row.get('Cliente', 'Desconocido'),
                        'Producto_Terminado': row.get('Producto Terminado', 'N/A'),
                        'Ciudad': row.get('Ciudad', 'N/A'),
                        'Componente': comp,
                        'Fecha_Entrega': row['Real'],
                        'Pedidas': pedidas,
                        'Surtidas_Sim': surtido,
                        "ProcesoSiguiente":row.get("Proceso","N/A")
                    })

            df_impacto_sim = pd.DataFrame(lineas_simuladas)

            # --- REPORTES EN CONSOLA POR ESCENARIO ---
            print("\n" + "="*50)
            print(f"   REPORTE DE INSIGHTS: ESCENARIO A {horas} HORAS")
            print("="*50)

            total_pzas_solicitadas = df_ventas_urgentes['Pzas'].sum()
            total_pzas_cortadas_turno = df_dentro_turno['Piezas_Simuladas'].sum()
            total_pzas_utiles_comercial = df_impacto_sim['Surtidas_Sim'].sum() if not df_impacto_sim.empty else 0

            wip_inutil_sobreproduccion = max(0, total_pzas_cortadas_turno - total_pzas_utiles_comercial)
            nivel_servicio_global = (total_pzas_utiles_comercial / total_pzas_solicitadas * 100) if total_pzas_solicitadas > 0 else 0

            print(f"📊 Demanda Crítica Total: {total_pzas_solicitadas:,} pzas")
            print(f"🏭 Piezas Físicas Cortadas: {total_pzas_cortadas_turno:,} pzas")
            print(f"✅ Piezas Útiles (Asignadas): {total_pzas_utiles_comercial:,} pzas")
            print(f"⚠️ WIP / Sobreproducción: {wip_inutil_sobreproduccion:,} pzas")
            print(f"📈 Nivel de Servicio Global: {nivel_servicio_global:.2f}%")

            # --- EXPORTACIÓN DE EXCEL POR ESCENARIO ---
            hoy_str = datetime.now().strftime('%Y%m%d_%H%M')
            nombre_excel = f"Impacto_Lineas_Simulador_{horas}hrs_{hoy_str}.xlsx"

            if not df_impacto_sim.empty:
                columnas_exportacion = ['Cliente', 'Producto_Terminado', 'Componente', 'Fecha_Entrega', 'Pedidas', 'Surtidas_Sim']
                df_export = df_impacto_sim[columnas_exportacion].copy()
                df_export['Fecha_Entrega'] = df_export['Fecha_Entrega'].dt.strftime('%Y-%m-%d')
                df_export.to_excel(nombre_excel, index=False)
                print(f"💾 Reporte exportado: {nombre_excel}")

                cobertura_cliente = df_impacto_sim.groupby('Cliente')['Surtidas_Sim'].sum().reset_index()
                cobertura_cliente.rename(columns={'Surtidas_Sim': f'Surtidas_{horas}h'}, inplace=True)
                df_plot_all = pd.merge(df_plot_all, cobertura_cliente, on='Cliente', how='left')
            else:
                print("⚠️ Ninguna pieza cubrió la demanda en este escenario.")
                df_plot_all[f'Surtidas_{horas}h'] = 0

            df_plot_all[f'Surtidas_{horas}h'] = df_plot_all[f'Surtidas_{horas}h'].fillna(0)

        # =========================================================
        # 4. CONSTRUCCIÓN DE LA GRÁFICA COMPARATIVA GLOBAL
        # =========================================================
        df_plot_grafico = df_plot_all.sort_values(by=['Demanda_Critica'], ascending=False).head(10)
        df_plot_grafico['Cliente'] = df_plot_grafico['Cliente'].astype(str)

        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=df_plot_grafico['Cliente'], y=df_plot_grafico['Demanda_Critica'],
            name='Demanda Crítica Requerida', marker_color='#d3d3d3', opacity=0.8, hoverinfo='x+y+name'
        ))

        for horas in horizontes:
            color = colores_escenarios.get(horas, '#1abc9c')
            fig.add_trace(go.Bar(
                x=df_plot_grafico['Cliente'], y=df_plot_grafico[f'Surtidas_{horas}h'],
                name=f'Cobertura a {horas} hrs', marker_color=color, opacity=0.9, hoverinfo='x+y+name'
            ))

        fig.update_layout(
            title=dict(text='Proyección de Cobertura (8, 16, 24 y 48 Hrs)', font=dict(size=20)),
            xaxis_title='Cliente / Cuenta (Top 10)', yaxis_title='Cantidad de Piezas',
            barmode='group', template='plotly_white',
            hovermode='x unified', legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        fig.show()

        # Retornamos el DataFrame consolidado para que lo use la Sección 5
        return df_plot_all

    except FileNotFoundError as e:
        print(f"Error: Archivo no encontrado. {e}")
        return None
    except Exception as e:
        print(f"Error inesperado en la simulación: {e}")
        return None

# ==============================================================================
# BLOQUE DE EJECUCIÓN DIARIA
# ==============================================================================
ARCHIVO_VENTAS = 'Reporte de Cortos_07.07_laser.xlsx'
ARCHIVO_PLAN_ERP = 'Plan de Corte_07.07_xComponente.xlsx'

ventas_maestro, ventas_urgentes = preparar_datos_ventas(ARCHIVO_VENTAS, dias_visibilidad=7)

# Definimos los horizontes en una variable para reutilizarla en la Sección 5
lista_horizontes = [10, 24, 48, 72, 96, 120]

# Guardamos el resultado de la función en una variable
df_resultados_globales = simular_cobertura_escenarios(
    archivo_plan_actual=ARCHIVO_PLAN_ERP,
    df_ventas=ventas_maestro,
    df_ventas_urgentes=ventas_urgentes,
    horizontes=lista_horizontes
)

# ==============================================================================
# 5. ANÁLISIS DETALLADO POR CLIENTE (SECCIÓN INTERACTIVA)
# ==============================================================================
print("\n" + "="*50)
print(" 5. ANÁLISIS DETALLADO POR CLIENTE")
print("="*50)

if df_resultados_globales is not None and not df_resultados_globales.empty:
    # Filtrar la lista para mostrar solo los 20 principales por Demanda_Critica
    df_top_clientes = df_resultados_globales.sort_values(by='Demanda_Critica', ascending=False).head(20)
    lista_clientes_top = df_top_clientes['Cliente'].unique().tolist()

    # Función encapsulada que se ejecutará cada vez que cambie el dropdown
    def graficar_cliente(cliente_seleccionado):
        df_cliente_seleccionado = df_resultados_globales[df_resultados_globales['Cliente'] == cliente_seleccionado].copy()

        if not df_cliente_seleccionado.empty:
            data_cliente = df_cliente_seleccionado.iloc[0]
            demanda_critica_cliente = data_cliente['Demanda_Critica']

            coberturas = []
            nombres_horizontes = []
            for horas in lista_horizontes:
                col_surtidas = f'Surtidas_{horas}h'
                if col_surtidas in data_cliente:
                    coberturas.append(data_cliente[col_surtidas])
                    nombres_horizontes.append(f'{horas}h')

            fig_cliente = go.Figure()

            # Demanda Crítica Base
            fig_cliente.add_trace(go.Bar(
                x=['Demanda Crítica'], y=[demanda_critica_cliente],
                name='Demanda Crítica Requerida',
                marker_color='#d3d3d3',
                opacity=0.8
            ))

            # Proyección Multi-Horizonte
            fig_cliente.add_trace(go.Bar(
                x=nombres_horizontes, y=coberturas,
                name='Cobertura Proyectada (Piezas Surtidas)',
                marker_color='#3498db',
                opacity=0.9
            ))

            fig_cliente.update_layout(
                title=dict(text=f'Proyección de Cobertura para Cliente: {cliente_seleccionado}', font=dict(size=20)),
                xaxis_title='Horizonte / Demanda',
                yaxis_title='Cantidad de Piezas',
                barmode='group',
                template='plotly_white',
                hovermode='x unified',
                legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
            )
            fig_cliente.show()
        else:
            print(f"No se encontraron datos para el cliente: {cliente_seleccionado}")

    # Crear el widget interactivo
    cliente_selector = widgets.Dropdown(options=lista_clientes_top, description='Cliente:')

    # La magia de interact: vincula la función graficar_cliente con el valor del dropdown
    widgets.interact(graficar_cliente, cliente_seleccionado=cliente_selector)

else:
    print("No se pudo cargar la sección interactiva porque la simulación no arrojó resultados.")

Iniciando Módulo de Simulación de Escenarios Multi-Horizonte...
✅ Ventas cargadas: 3237 líneas críticas detectadas.

Arrancando simulación de reloj maestro para escenarios de [10, 24, 48, 72, 96, 120] horas...

   REPORTE DE INSIGHTS: ESCENARIO A 10 HORAS
📊 Demanda Crítica Total: 173,931 pzas
🏭 Piezas Físicas Cortadas: 9,475 pzas
✅ Piezas Útiles (Asignadas): 8,047.0 pzas
⚠️ WIP / Sobreproducción: 1,428.0 pzas
📈 Nivel de Servicio Global: 4.63%
💾 Reporte exportado: Impacto_Lineas_Simulador_10hrs_20260709_1947.xlsx

   REPORTE DE INSIGHTS: ESCENARIO A 24 HORAS
📊 Demanda Crítica Total: 173,931 pzas
🏭 Piezas Físicas Cortadas: 19,645 pzas
✅ Piezas Útiles (Asignadas): 14,746.0 pzas
⚠️ WIP / Sobreproducción: 4,899.0 pzas
📈 Nivel de Servicio Global: 8.48%
💾 Reporte exportado: Impacto_Lineas_Simulador_24hrs_20260709_1947.xlsx

   REPORTE DE INSIGHTS: ESCENARIO A 48 HORAS
📊 Demanda Crítica Total: 173,931 pzas
🏭 Piezas Físicas Cortadas: 38,099 pzas
✅ Piezas Útiles (Asignadas): 30,060.0 pzas
⚠️ WIP


 5. ANÁLISIS DETALLADO POR CLIENTE


interactive(children=(Dropdown(description='Cliente:', options=('Caterpillar Inc.', 'INTERNATIONAL MOTORS MEXI…